# Video speech and visual content

Pause the web app while running GPU notebooks. These cells use the same modules as FastAPI.


Lab 4 local cells 4–8 read precomputed descriptions and timestamps; they do not process raw video. This notebook demonstrates the new raw-video adapter, reusing Lab 2 transcription and L6 OCR/VLM components.


In [ ]:
from pathlib import Path
import sys
# Find the standalone project whether Jupyter starts at its root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "app/config.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from app.config import Settings
from app.processors import Processor
from app.files import atomic_text, save_json
settings = Settings(root=ROOT)
settings.initialize()
processor = Processor(settings, progress=print)



In [ ]:
from app.config import FORMATS
candidates = sorted(p for p in (ROOT / "input/video").rglob("*") if p.suffix.lower() in FORMATS["video"])
if not candidates:
    raise FileNotFoundError("Add your own MP4/MKV/MOV under input/video/. No raw video is supplied in the original dataset.")
source = candidates[0]
print(source)



Transcribe the entire audio track, then sample visuals every 10 seconds. Silent videos still receive frame analysis. Sampling is explicit and does not claim exhaustive frame coverage.


In [ ]:
settings.frame_interval = 10
result = processor.process(source, "video")
print("Review notes:", result.warnings)



In [ ]:
# Write one text output with its original extension and relative path preserved.
relative = source.relative_to(ROOT / "input")
output = ROOT / "output" / (relative.as_posix() + ".txt")
atomic_text(output, result.text(relative.as_posix()))
save_json(ROOT / "runtime" / "notebook-result.json", {
    "source": str(relative), "warnings": result.warnings,
    "devices": result.devices, "details": result.details,
})
print(output)
print(output.read_text(encoding="utf-8")[:5000])

